In [4]:
from dotenv import load_dotenv
import os
from pathlib import Path
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

## Load test dataset

In [5]:
# Shuffle
test_df = pd.read_csv('../Splits/example_test/Mixed.csv')
test_df

,Unnamed: 0.1,writer,problem_id,submission_id,website,source,Unnamed: 0
0,0,AI,164A,NaN,https://codeforces.com,#include <bits/stdc++.h>\nusing namespace std;...,NaN
1,1,AI,NaN,NaN,https://codeforces.com,#include <bits/stdc++.h>\nusing namespace std;...,NaN
2,2,AI,o61_may08_estate,NaN,https://programming.in.th,#include <bits/stdc++.h>\nusing namespace std;...,1843.0
3,3,human,NaN,13646458.0,https://codeforces.com,//Template\n\n// By Anudeep :)\n//Includes\n#i...,NaN
4,4,AI,1032,NaN,https://programming.in.th,#include <bits/stdc++.h>\nusing namespace std;...,998.0
...,...,...,...,...,...,...,...
2995,2995,human,1000,306681.0,https://programming.in.th,#include <bits/stdc++.h>\nusing namespace std;...,NaN
2996,2996,human,1055,310229.0,https://programming.in.th,#include <bits/stdc++.h>\nusing namespace std;...,NaN
2997,2997,Human,NaN,NaN,https://firefly.gchan.moe,#include <bits/stdc++.h>\n\nusing namespace st...,NaN
2998,2998,AI,888F,NaN,https://codeforces.com,#include <bits/stdc++.h>\nusing namespace std;...,NaN


In [6]:
test_df['source'][0]

'#include <bits/stdc++.h>\nusing namespace std;\n\nint main() {\n    ios_base::sync_with_stdio(false);\n    cin.tie(nullptr);\n\n    int n, m;\n    cin >> n >> m;\n\n    vector<int> f(n);\n    for (int& x : f) cin >> x;\n\n    vector<vector<int>> adj(n + 1), reverse_adj(n + 1);\n    for (int i = 0; i < m; ++i) {\n        int a, b;\n        cin >> a >> b;\n        adj[a].push_back(b);\n        reverse_adj[b].push_back(a);\n    }\n\n    vector<bool> forward(n + 1, false), backward(n + 1, false);\n    queue<int> q;\n\n    // Compute forward reachable (from assigns through 0s and 2s)\n    for (int u = 1; u <= n; ++u) {\n        if (f[u - 1] == 1 && !forward[u]) {\n            forward[u] = true;\n            q.push(u);\n        }\n    }\n    while (!q.empty()) {\n        int u = q.front();\n        q.pop();\n        for (int v : adj[u]) {\n            if (!forward[v]) {\n                if (f[v - 1] == 0) {\n                    forward[v] = true;\n                    q.push(v);\n           

In [7]:
url = "http://localhost:59875/analyze"
response = requests.post(url, json={"code":test_df['source'][0]})
" ".join(json.loads(response.text)['details'])

'Consistent formatting detected (uniform indentation), which is often a trait of AI-generated code. Specific, context-aware variable names detected, which is more typical of human-written code. Comments found, indicating that the code might have been written or reviewed by a human. Repetitive patterns found in the code, which can be a sign of AI generation. Overly structured code detected (e.g., repeated if-else or loops), which might indicate AI involvement. Simpler code detected, which could indicate AI generation.'

In [8]:
responses = []
err_logs = []
url = "http://localhost:59875/analyze"
from rich import print

from IPython.display import clear_output
for index, row in test_df.iterrows():
  source = row['source']
  print(f"ASKED : {index}")
  try:
    response = requests.post(url, json={"code":source})
    clear_output()
    data = json.loads(response.text)
    responses.append(data)
  except Exception as e:
    clear_output()
    err_logs.append((index,e))
    print(f"ERROR at {index} : {e}")

  

In [9]:
err_logs

[]

In [11]:
import re
import json

result_n = len(responses)
answer_df = test_df.iloc[:result_n]
answer_df['answer_text'] = responses

C:\Users\Pakin\AppData\Local\Temp\ipykernel_20884\2693926148.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  answer_df['answer_text'] = responses


In [12]:
answer_df['model_answer'] = answer_df['answer_text'].apply(lambda x: x['evaluate'])
answer_df['model_reason'] = answer_df['answer_text'].apply(lambda x: " ".join(x['details']))
answer_df['verdict'] = answer_df['writer'] == answer_df['model_answer']
answer_df = answer_df[['writer','model_answer','verdict','model_reason','website','source']]

C:\Users\Pakin\AppData\Local\Temp\ipykernel_20884\319285290.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  answer_df['model_answer'] = answer_df['answer_text'].apply(lambda x: x['evaluate'])
C:\Users\Pakin\AppData\Local\Temp\ipykernel_20884\319285290.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  answer_df['model_reason'] = answer_df['answer_text'].apply(lambda x: " ".join(x['details']))
C:\Users\Pakin\AppData\Local\Temp\ipykernel_20884\319285290.py:3: SettingWithCopyWarning: 
A value is trying to

In [13]:
filtered_df = pd.DataFrame()
filtered_df[['Label','Model_Answer','Verdict','Reason','Website']] = answer_df[['writer','model_answer','verdict','model_reason','website']]

In [21]:
# I forgot about upper-lower case T-T
filtered_df['Label'] = filtered_df['Label'].replace('Human', 'human')
filtered_df['Model_Answer'] = filtered_df['Model_Answer'].replace('Human', 'human')
filtered_df['Verdict'] = filtered_df['Label']==filtered_df['Model_Answer']

In [22]:
filtered_df['Label'].value_counts(), filtered_df['Model_Answer'].value_counts()

(Label
 AI       1500
 human    1500
 Name: count, dtype: int64,
 Model_Answer
 AI       1972
 human    1028
 Name: count, dtype: int64)

In [23]:
filtered_df.to_csv("./Results/_2/Result-RuleBased.csv",index=False)

In [24]:
accuracy = filtered_df['Verdict'].value_counts(normalize=True).get(True, 0)
print(f"AUC : {accuracy}")

AUC : 0.568